<a href="https://colab.research.google.com/github/l3ryx9/Volt/blob/main/VoltAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# [1/8] 🔎 DÉTECTION GPU + PRÉPARATION
# ============================================================

import os
import sys
import time
import shutil
import subprocess

from tqdm.auto import tqdm

print()
print("=" * 70)
print("🚀 [1/8] 🔎 DÉTECTION GPU + PRÉPARATION")
print("=" * 70)

start = time.time()

# ------------------------------------------------------------
# Vérification GPU
# ------------------------------------------------------------

print("\n🔍 Vérification PyTorch...")

try:
    import torch
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "torch"],
        check=True
    )
    import torch

print(f"   PyTorch : {torch.__version__}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ CUDA indisponible. Active un GPU dans Colab."
    )

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)

vram = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print(f"   ✅ GPU : {gpu_name}")
print(
    f"   🧩 Compute Capability : "
    f"sm_{capability[0]}{capability[1]}"
)
print(f"   💾 VRAM : {vram:.2f} Go")

if capability == (7, 5):
    print("   ✅ Tesla T4 / SM75 détectée")
else:
    print("   ⚠️ GPU différent de Tesla T4")

# ------------------------------------------------------------
# CUDA
# ------------------------------------------------------------

print("\n🔧 Vérification CUDA...")

nvcc = shutil.which("nvcc")

if nvcc:
    result = subprocess.run(
        [nvcc, "--version"],
        capture_output=True,
        text=True
    )

    for line in result.stdout.splitlines():
        if "release" in line.lower():
            print(f"   {line.strip()}")
            break
else:
    print("   ⚠️ nvcc introuvable")

# ------------------------------------------------------------
# Dossiers locaux
# ------------------------------------------------------------

ROOT = "/content"
LOCAL_LLAMA = f"{ROOT}/llama.cpp"
LOCAL_MODELS = f"{ROOT}/models"

os.makedirs(LOCAL_MODELS, exist_ok=True)

print("\n📁 Dossiers locaux préparés")

print(f"   llama.cpp : {LOCAL_LLAMA}")
print(f"   models    : {LOCAL_MODELS}")

print()
print("=" * 70)
print(
    f"✅ [1/8] TERMINÉ — "
    f"{time.time() - start:.1f}s"
)
print("=" * 70)


🚀 [1/8] 🔎 DÉTECTION GPU + PRÉPARATION

🔍 Vérification PyTorch...
   PyTorch : 2.11.0+cu128
   ✅ GPU : Tesla T4
   🧩 Compute Capability : sm_75
   💾 VRAM : 14.56 Go
   ✅ Tesla T4 / SM75 détectée

🔧 Vérification CUDA...
   Cuda compilation tools, release 12.8, V12.8.93

📁 Dossiers locaux préparés
   llama.cpp : /content/llama.cpp
   models    : /content/models

✅ [1/8] TERMINÉ — 0.0s


In [ ]:
# ============================================================
# [2/8] 💾 GOOGLE DRIVE — STOCKAGE PERSISTANT
# ============================================================

import os
import time
import shutil

from google.colab import drive

print()
print("=" * 70)
print("🚀 [2/8] 💾 GOOGLE DRIVE — STOCKAGE PERSISTANT")
print("=" * 70)

start = time.time()

print("\n🔗 Connexion à Google Drive...")

drive.mount(
    "/content/drive",
    force_remount=False
)

DRIVE_ROOT = "/content/drive/MyDrive/Qwen-Colab"

DRIVE_LLAMA = f"{DRIVE_ROOT}/llama.cpp"
DRIVE_MODELS = f"{DRIVE_ROOT}/models"
DRIVE_CACHE = f"{DRIVE_ROOT}/cache"

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DRIVE_LLAMA, exist_ok=True)
os.makedirs(DRIVE_MODELS, exist_ok=True)
os.makedirs(DRIVE_CACHE, exist_ok=True)

print("\n   ✅ Google Drive connecté")

print()
print("📁 Stockage persistant :")

print(f"   ├─ {DRIVE_ROOT}")
print(f"   ├─ llama.cpp → {DRIVE_LLAMA}")
print(f"   ├─ models    → {DRIVE_MODELS}")
print(f"   └─ cache     → {DRIVE_CACHE}")

print()
print("=" * 70)
print(
    f"✅ [2/8] TERMINÉ — "
    f"{time.time() - start:.1f}s"
)
print("=" * 70)


🚀 [2/8] 💾 GOOGLE DRIVE — STOCKAGE PERSISTANT

🔗 Connexion à Google Drive...
Mounted at /content/drive

   ✅ Google Drive connecté

📁 Stockage persistant :
   ├─ /content/drive/MyDrive/Qwen-Colab
   ├─ llama.cpp → /content/drive/MyDrive/Qwen-Colab/llama.cpp
   ├─ models    → /content/drive/MyDrive/Qwen-Colab/models
   └─ cache     → /content/drive/MyDrive/Qwen-Colab/cache

✅ [2/8] TERMINÉ — 28.1s


In [ ]:
# ============================================================
# [3/8] 🦙 LLAMA.CPP CUDA SM75 — CACHE PERSISTANT
# ============================================================

import os
import time
import shutil
import subprocess

print()
print("=" * 70)
print("🚀 [3/8] 🦙 LLAMA.CPP CUDA SM75")
print("=" * 70)

start = time.time()

LOCAL_LLAMA = "/content/llama.cpp"
LOCAL_BUILD = f"{LOCAL_LLAMA}/build"

DRIVE_LLAMA = "/content/drive/MyDrive/Qwen-Colab/llama.cpp"
DRIVE_BUILD = f"{DRIVE_LLAMA}/build"

SERVER_LOCAL = f"{LOCAL_BUILD}/bin/llama-server"
SERVER_DRIVE = f"{DRIVE_BUILD}/bin/llama-server"


# ============================================================
# Recherche du serveur
# ============================================================

def find_server(path):

    if os.path.isfile(path):
        return path

    if not os.path.isdir(path):
        return None

    for root, dirs, files in os.walk(path):

        if "llama-server" in files:

            candidate = os.path.join(
                root,
                "llama-server"
            )

            if os.access(candidate, os.X_OK):
                return candidate

    return None


print("\n🔍 Recherche du llama-server persistant...")

persistent_server = find_server(DRIVE_LLAMA)

if persistent_server:

    print(
        f"   ✅ llama-server trouvé dans Drive :"
    )

    print(
        f"   {persistent_server}"
    )

    print(
        "\n⚡ Pas de recompilation nécessaire."
    )

    # Copie du projet vers /content
    if os.path.exists(LOCAL_LLAMA):
        shutil.rmtree(LOCAL_LLAMA)

    print("\n📥 Copie de llama.cpp vers /content...")

    shutil.copytree(
        DRIVE_LLAMA,
        LOCAL_LLAMA
    )

else:

    print(
        "   ⚠️ Aucun build persistant trouvé."
    )

    print(
        "\n📥 Récupération de llama.cpp..."
    )

    if not os.path.isdir(LOCAL_LLAMA):

        subprocess.run(
            [
                "git",
                "clone",
                "--depth=1",
                "https://github.com/ggml-org/llama.cpp.git",
                LOCAL_LLAMA
            ],
            check=True
        )

    print("\n⚙️ Configuration CUDA SM75...")

    subprocess.run(
        [
            "cmake",
            "-S",
            LOCAL_LLAMA,
            "-B",
            LOCAL_BUILD,
            "-DGGML_CUDA=ON",
            "-DCMAKE_CUDA_ARCHITECTURES=75",
            "-DLLAMA_CURL=OFF",
            "-DCMAKE_BUILD_TYPE=Release"
        ],
        check=True
    )

    print("\n🏗️ Compilation en temps réel...")

    process = subprocess.Popen(
        [
            "cmake",
            "--build",
            LOCAL_BUILD,
            "--config",
            "Release",
            "-j2"
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:

        line = line.rstrip()

        if line:
            print(
                f"   {line}",
                flush=True
            )

    process.wait()

    if process.returncode != 0:
        raise RuntimeError(
            "❌ Compilation llama.cpp échouée."
        )

    print("\n💾 Sauvegarde du build dans Drive...")

    if os.path.exists(DRIVE_LLAMA):
        shutil.rmtree(DRIVE_LLAMA)

    shutil.copytree(
        LOCAL_LLAMA,
        DRIVE_LLAMA
    )

    print("   ✅ Build sauvegardé.")


# ============================================================
# Recherche finale
# ============================================================

SERVER = find_server(LOCAL_LLAMA)

if not SERVER:

    raise FileNotFoundError(
        "❌ llama-server introuvable après préparation."
    )

os.chmod(
    SERVER,
    0o755
)

print()
print("🦙 llama-server prêt :")
print(f"   {SERVER}")

print()
print("🧪 Test...")

subprocess.run(
    [
        SERVER,
        "--version"
    ],
    check=True
)

print()
print("=" * 70)
print(
    f"✅ [3/8] LLAMA.CPP PRÊT — "
    f"{time.time() - start:.1f}s"
)
print("=" * 70)


🚀 [3/8] 🦙 LLAMA.CPP CUDA SM75

🔍 Recherche du llama-server persistant...
   ⚠️ Aucun build persistant trouvé.

📥 Récupération de llama.cpp...

⚙️ Configuration CUDA SM75...

🏗️ Compilation en temps réel...
   [  0%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/hash.cpp.o
   [  0%] Building C object ggml/src/CMakeFiles/ggml-base.dir/ggml.c.o
   [  0%] Building C object vendor/hash/CMakeFiles/vendor-hash.dir/xxhash/xxhash.c.o
   [  0%] Building CXX object vendor/hash/CMakeFiles/vendor-hash.dir/sha1/sha1.c.o
   [  0%] Building C object vendor/hash/CMakeFiles/vendor-hash.dir/sha256/sha256.c.o
   [  1%] Linking CXX static library libvendor-hash.a
   [  1%] Built target vendor-hash
   [  1%] Building CXX object common/CMakeFiles/llama-common-base.dir/build-info.cpp.o
   [  1%] Linking CXX static library libllama-common-base.a
   [  1%] Built target llama-common-base
   [  1%] Building CXX object vendor/cpp-httplib/CMakeFiles/cpp-httplib.dir/httplib.cpp.o
   [  2%] Building CXX

In [ ]:
# ============================================================
# [4/8] 🧠 QWEN 2.5 CODER 7B UNCENSORED Q5_K_M
# ============================================================

import os
import time
import subprocess
import sys

print()
print("=" * 70)
print("🚀 [4/8] 🧠 QWEN 2.5 CODER 7B UNCENSORED Q5_K_M")
print("=" * 70)

start = time.time()

DRIVE_ROOT = "/content/drive/MyDrive/Qwen-Colab"

DRIVE_MODELS = f"{DRIVE_ROOT}/models"
LOCAL_MODELS = "/content/models"

os.makedirs(DRIVE_MODELS, exist_ok=True)
os.makedirs(LOCAL_MODELS, exist_ok=True)


# ============================================================
# MODÈLE CIBLE
# ============================================================

HF_REPO = (
    "BlossomsAI/"
    "Qwen2.5-Coder-7B-Instruct-Uncensored-GGUF"
)

MODEL_FILE = "q5_k_m.gguf"

DRIVE_MODEL = (
    f"{DRIVE_MODELS}/{MODEL_FILE}"
)

LOCAL_MODEL = (
    f"{LOCAL_MODELS}/{MODEL_FILE}"
)


# ============================================================
# [1/5] VÉRIFICATION DRIVE
# ============================================================

print()
print("🔍 [1/5] Vérification du cache Google Drive...")

if os.path.isfile(DRIVE_MODEL):

    size = (
        os.path.getsize(DRIVE_MODEL)
        / 1024**3
    )

    print(
        f"   ✅ Modèle trouvé dans Drive"
    )

    print(
        f"   📦 {size:.2f} Go"
    )

    print(
        "   ⚡ Aucun téléchargement nécessaire."
    )

else:

    print(
        "   ❌ Modèle absent du cache."
    )

    print()
    print(
        "📥 Téléchargement depuis Hugging Face..."
    )

    print(
        f"   Repository : {HF_REPO}"
    )

    print(
        f"   Fichier    : {MODEL_FILE}"
    )

    print()

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "huggingface_hub"
        ],
        check=True
    )

    # --------------------------------------------------------
    # Téléchargement avec progression HF
    # --------------------------------------------------------

    subprocess.run(
        [
            "hf",
            "download",
            HF_REPO,
            MODEL_FILE,
            "--local-dir",
            DRIVE_MODELS
        ],
        check=True
    )

    if not os.path.isfile(DRIVE_MODEL):

        raise RuntimeError(
            "❌ Le téléchargement du modèle a échoué."
        )

    print()
    print(
        "   ✅ Téléchargement terminé."
    )


# ============================================================
# [2/5] TAILLE
# ============================================================

print()
print("📦 [2/5] Vérification du fichier...")

size = (
    os.path.getsize(DRIVE_MODEL)
    / 1024**3
)

print(
    f"   Fichier : {MODEL_FILE}"
)

print(
    f"   Taille  : {size:.2f} Go"
)


# ============================================================
# [3/5] COPIE VERS /CONTENT
# ============================================================

print()
print("📥 [3/5] Préparation du modèle local...")

if os.path.isfile(LOCAL_MODEL):

    local_size = (
        os.path.getsize(LOCAL_MODEL)
        / 1024**3
    )

    print(
        f"   ✅ Déjà présent : "
        f"{local_size:.2f} Go"
    )

else:

    print(
        "   📥 Copie depuis Google Drive..."
    )

    subprocess.run(
        [
            "cp",
            DRIVE_MODEL,
            LOCAL_MODEL
        ],
        check=True
    )

    print(
        "   ✅ Copie terminée."
    )


# ============================================================
# [4/5] VALIDATION
# ============================================================

print()
print("🧪 [4/5] Validation du GGUF...")

if not os.path.isfile(LOCAL_MODEL):

    raise FileNotFoundError(
        f"❌ Modèle introuvable : {LOCAL_MODEL}"
    )

final_size = (
    os.path.getsize(LOCAL_MODEL)
    / 1024**3
)

if final_size < 5.0:

    raise RuntimeError(
        f"""
❌ Fichier GGUF suspect.

Taille détectée :
{final_size:.2f} Go

Le Q5_K_M attendu fait environ 5,44 Go.
"""
    )

print(
    f"   ✅ GGUF valide : "
    f"{final_size:.2f} Go"
)


# ============================================================
# [5/5] CONFIGURATION
# ============================================================

MODEL = LOCAL_MODEL

print()
print("=" * 70)
print("🎯 MODÈLE SÉLECTIONNÉ")
print("=" * 70)

print()
print(
    "🧠 Qwen2.5-Coder-7B-Instruct-Uncensored"
)

print(
    "⚙️ Quantification : Q5_K_M"
)

print(
    f"📦 Fichier : {MODEL}"
)

print(
    f"💾 Taille : {final_size:.2f} Go"
)

print()
print(
    "🌐 Source :"
)

print(
    "   BlossomsAI/Qwen2.5-Coder-7B-Instruct-Uncensored-GGUF"
)

print()
print("=" * 70)

print(
    f"✅ [4/8] QWEN UNCENSORED PRÊT — "
    f"{time.time() - start:.1f}s"
)

print("=" * 70)

In [ ]:
# ============================================================
# [5/8] 🚀 DÉMARRAGE DE QWEN SUR LA T4
# ============================================================

import os
import time
import signal
import socket
import subprocess
import urllib.request

print()
print("=" * 70)
print("🚀 [5/8] DÉMARRAGE DE QWEN SUR LA T4")
print("=" * 70)

HOST = "127.0.0.1"
PORT = 8080

LOG_FILE = "/content/llama-server.log"

# SERVER vient de [3/8]
# MODEL vient de [4/8]

if not os.path.isfile(SERVER):
    raise FileNotFoundError(
        f"❌ llama-server absent : {SERVER}"
    )

if not os.path.isfile(MODEL):
    raise FileNotFoundError(
        f"❌ Modèle absent : {MODEL}"
    )


# ============================================================
# PORT
# ============================================================

def port_used(port):

    sock = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM
    )

    sock.settimeout(0.3)

    try:
        return sock.connect_ex(
            (HOST, port)
        ) == 0
    finally:
        sock.close()


print("\n🔌 Vérification du port 8080...")

if port_used(PORT):

    print(
        "   ⚠️ Port 8080 déjà utilisé."
    )

    result = subprocess.run(
        [
            "fuser",
            "-n",
            "tcp",
            str(PORT)
        ],
        capture_output=True,
        text=True
    )

    pids = [
        int(x)
        for x in result.stdout.split()
        if x.isdigit()
    ]

    for pid in pids:

        try:

            print(
                f"   🛑 Arrêt PID {pid}"
            )

            os.kill(
                pid,
                signal.SIGTERM
            )

        except:
            pass

    for i in range(20):

        if not port_used(PORT):
            break

        time.sleep(1)

    if port_used(PORT):

        raise RuntimeError(
            "❌ Impossible de libérer le port 8080."
        )

print(
    "   ✅ Port 8080 disponible"
)


# ============================================================
# LOG
# ============================================================

if os.path.exists(LOG_FILE):
    os.remove(LOG_FILE)


# ============================================================
# COMMANDE
# ============================================================

command = [
    SERVER,

    "--host",
    HOST,

    "--port",
    str(PORT),

    "--model",
    MODEL,

    "--n-gpu-layers",
    "999",

    "--ctx-size",
    "32768",

    "--cache-type-k",
    "q8_0",

    "--cache-type-v",
    "q8_0",

    "--n-predict",
    "-1",

    "--alias",
    "qwen2.5-coder-7b-instruct-uncensored",

    "--alias",
    "q5_k_m",

    "--alias",
    "qwen2.5-coder-7b-instruct-q5_k_m",

    "--threads",
    "2",

    "--log-file",
    LOG_FILE
]

print()
print("⚙️ Configuration :")
print(f"   ├─ GPU          : Tesla T4 / SM75")
print(f"   ├─ Modèle       : {MODEL}")
print(f"   ├─ GPU layers   : 999")
print(f"   ├─ Contexte     : 32768")
print(f"   ├─ Tokens       : illimités (-1)")
print(f"   ├─ Alias        : qwen2.5-coder-7b-instruct-uncensored")
print(f"   └─ API          : http://{HOST}:{PORT}")


# ============================================================
# START
# ============================================================

print()
print("🚀 Lancement de llama-server...")

SERVER_PROCESS = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print(
    f"   ✅ PID : {SERVER_PROCESS.pid}"
)


# ============================================================
# HEALTH + PROGRESSION
# ============================================================

print()
print("📡 Chargement de Qwen...")

MAX_WAIT = 300

ready = False

for elapsed in range(MAX_WAIT):

    if SERVER_PROCESS.poll() is not None:

        print()

        print(
            f"❌ llama-server arrêté "
            f"(code {SERVER_PROCESS.returncode})"
        )

        if os.path.exists(LOG_FILE):

            print(
                open(
                    LOG_FILE,
                    "r",
                    errors="replace"
                ).read()[-12000:]
            )

        raise RuntimeError(
            "❌ Qwen n'a pas démarré."
        )


    try:

        response = urllib.request.urlopen(
            f"http://{HOST}:{PORT}/health",
            timeout=1
        )

        if response.status == 200:

            ready = True

            print()

            print(
                f"   🚀 Qwen prêt après "
                f"{elapsed + 1}s"
            )

            break

    except:
        pass


    percent = int(
        ((elapsed + 1) / MAX_WAIT) * 100
    )

    width = 40

    filled = int(
        width * percent / 100
    )

    bar = (
        "█" * filled
        + "░" * (width - filled)
    )

    print(
        f"\r   [{bar}] "
        f"{percent:3d}% "
        f"| {elapsed + 1:3d}s/{MAX_WAIT}s",
        end="",
        flush=True
    )

    time.sleep(1)


if not ready:

    raise RuntimeError(
        "❌ Qwen n'est pas devenu disponible."
    )


print()
print()
print("=" * 70)
print("✅ [5/8] QWEN PRÊT")
print("=" * 70)

print()
print(f"🌐 API locale : http://{HOST}:{PORT}")
print(f"❤️ Health     : http://{HOST}:{PORT}/health")
print(f"🔢 PID        : {SERVER_PROCESS.pid}")


🚀 [5/8] DÉMARRAGE DE QWEN SUR LA T4


NameError: name 'SERVER' is not defined

In [ ]:
# ============================================================
# [6/8] 🌐 TUNNEL CLOUDFLARE
# ============================================================

import os
import re
import time
import subprocess

print()
print("=" * 70)
print("🚀 [6/8] 🌐 CRÉATION DU TUNNEL")
print("=" * 70)

start = time.time()

PORT = 8080

# ------------------------------------------------------------
# Installation cloudflared
# ------------------------------------------------------------

cloudflared = "/usr/local/bin/cloudflared"

if not os.path.isfile(cloudflared):

    print("\n📥 Installation de cloudflared...")

    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            cloudflared,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        ],
        check=True
    )

    os.chmod(
        cloudflared,
        0o755
    )

else:

    print(
        "\n   ✅ cloudflared déjà installé"
    )


# ------------------------------------------------------------
# Lancement tunnel
# ------------------------------------------------------------

print("\n🚇 Création du tunnel public...")

TUNNEL_LOG = "/content/cloudflared.log"

if os.path.exists(TUNNEL_LOG):
    os.remove(TUNNEL_LOG)

tunnel = subprocess.Popen(
    [
        cloudflared,
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}",
        "--no-autoupdate"
    ],
    stdout=open(TUNNEL_LOG, "w"),
    stderr=subprocess.STDOUT
)

print(
    f"   PID tunnel : {tunnel.pid}"
)


# ------------------------------------------------------------
# Recherche URL
# ------------------------------------------------------------

print()
print("⏳ Attente de l'URL publique...")

PUBLIC_URL = None

for i in range(60):

    time.sleep(1)

    if os.path.exists(TUNNEL_LOG):

        with open(
            TUNNEL_LOG,
            "r",
            errors="replace"
        ) as f:

            text = f.read()

        matches = re.findall(
            r"https://[-a-zA-Z0-9]+\.trycloudflare\.com",
            text
        )

        if matches:

            PUBLIC_URL = matches[0]
            break


    print(
        f"\r   ⏳ {i + 1}/60s",
        end="",
        flush=True
    )


print()

if not PUBLIC_URL:

    print(
        open(
            TUNNEL_LOG,
            "r",
            errors="replace"
        ).read()[-5000:]
    )

    raise RuntimeError(
        "❌ URL Cloudflare introuvable."
    )


print()
print("=" * 70)
print("🌐 TUNNEL ACTIF")
print("=" * 70)

print()
print(
    f"🔗 URL publique : {PUBLIC_URL}"
)

print(
    f"❤️ Health       : "
    f"{PUBLIC_URL}/health"
)

print()
print(
    f"⏱️ Temps : "
    f"{time.time() - start:.1f}s"
)

print("=" * 70)

In [ ]:
# ============================================================
# [7/8] ❤️ TEST API QWEN
# ============================================================

import json
import time
import urllib.request

print()
print("=" * 70)
print("🚀 [7/8] ❤️ TEST API QWEN")
print("=" * 70)

start = time.time()

# PUBLIC_URL vient de [6/8]

if "PUBLIC_URL" not in globals():
    raise RuntimeError(
        "❌ PUBLIC_URL absent. Exécute la cellule [6/8]."
    )


# ============================================================
# HEALTH
# ============================================================

print("\n❤️ Test /health...")

try:

    response = urllib.request.urlopen(
        f"{PUBLIC_URL}/health",
        timeout=10
    )

    body = response.read().decode(
        "utf-8",
        errors="replace"
    )

    print(
        f"   ✅ HTTP {response.status}"
    )

    print(
        f"   📡 {body}"
    )

except Exception as e:

    raise RuntimeError(
        f"❌ Health check échoué : {e}"
    )


# ============================================================
# MODELS
# ============================================================

print("\n🧠 Test /v1/models...")

try:

    response = urllib.request.urlopen(
        f"{PUBLIC_URL}/v1/models",
        timeout=10
    )

    body = response.read().decode(
        "utf-8",
        errors="replace"
    )

    data = json.loads(body)

    print(
        "   ✅ API OpenAI-compatible disponible"
    )

    print(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False
        )[:4000]
    )

except Exception as e:

    print(
        f"   ⚠️ /v1/models : {e}"
    )


# ============================================================
# FIN
# ============================================================

print()
print("=" * 70)
print(
    f"✅ [7/8] API VALIDÉE — "
    f"{time.time() - start:.1f}s"
)
print("=" * 70)

In [ ]:
# ============================================================
# [8/8] 🎉 QWEN COLAB PRÊT
# ============================================================

import json
import os

print()
print("=" * 70)
print("🚀 [8/8] 🎉 QWEN COLAB PRÊT")
print("=" * 70)

print()
print("╔════════════════════════════════════════════════════════════╗")
print("║                    QWEN 2.5 CODER 7B                     ║")
print("╚════════════════════════════════════════════════════════════╝")

print()
print("🎮 GPU")
print(f"   {gpu_name}")
print(
    f"   SM_{capability[0]}{capability[1]}"
)
print(
    f"   VRAM : {vram:.2f} Go"
)

print()
print("🧠 MODÈLE")
print(
    f"   {MODEL}"
)

print()
print("🦙 LLAMA.CPP")
print(
    f"   {SERVER}"
)

print()
print("🌐 API LOCALE")
print(
    "   http://127.0.0.1:8080"
)

print()
print("🌍 API PUBLIQUE")
print(
    f"   {PUBLIC_URL}"
)

print()
print("❤️ HEALTH")
print(
    f"   {PUBLIC_URL}/health"
)

print()
print("💾 CACHE PERSISTANT")
print(
    "   /content/drive/MyDrive/Qwen-Colab/"
)

print()
print("=" * 70)
print("📋 CONFIGURATION API")
print("=" * 70)

config = {
    "provider": "local-llama-cpp",
    "base_url": f"{PUBLIC_URL}/v1",
    "model": "qwen2.5-coder-7b-instruct-uncensored",
    "api_key": "",
    "gpu": "Tesla T4",
    "cuda_arch": "sm75",
    "openai_compatible": True
}

print(
    json.dumps(
        config,
        indent=2,
        ensure_ascii=False
    )
)

print()
print("=" * 70)
print("✅ [8/8] TERMINÉ")
print("=" * 70)

print()
print("💡 IMPORTANT")
print(
    "Les fichiers lourds sont maintenant persistants "
    "dans Google Drive."
)

print(
    "Après une nouvelle VM Colab, les cellules "
    "[3/8] et [4/8] réutiliseront le cache."
)

print()
print("⚠️ Le tunnel Cloudflare est temporaire :")
print(
    "une nouvelle session Colab donnera une nouvelle URL."
)

print("=" * 70)

In [ ]:
# ============================================================
# [9/9] ❤️ KEEP-ALIVE — SESSION COLAB PERSISTANTE
# ============================================================
# Boucle bloquante au premier plan : Colab considère le runtime
# occupé (pas de coupure pour inactivité) et les mini-requêtes
# /health maintiennent le tunnel Cloudflare + llama-server actifs
# pour ton application.
#
# ⚠️ Laisse cette cellule tourner en continu ; elle bloque les
# autres cellules tant qu'elle est active (arrêt : ▢ / Ctrl-C).
#
import random
import time
import urllib.request

if "PUBLIC_URL" in globals() and PUBLIC_URL:
    KEEPALIVE_URL = f"{PUBLIC_URL}/health"
else:
    KEEPALIVE_URL = f"http://{HOST}:{PORT}/health"

KEEPALIVE_MIN = 25
KEEPALIVE_MAX = 90

print()
print("=" * 70)
print("❤️ [9/9] KEEP-ALIVE ACTIF — SESSION MAINTENUE")
print("=" * 70)
print(f"   Cible      : {KEEPALIVE_URL}")
print(f"   Intervalle : {KEEPALIVE_MIN}-{KEEPALIVE_MAX}s (aléatoire)")
print("=" * 70)
print()

pings = 0

while True:
    time.sleep(random.randint(KEEPALIVE_MIN, KEEPALIVE_MAX))
    pings += 1

    try:
        r = urllib.request.urlopen(KEEPALIVE_URL, timeout=10)
        ok = r.status == 200
    except Exception as e:
        ok = False
        err = str(e)

    stamp = time.strftime("%H:%M:%S")
    if ok:
        print(f"   ✅ {stamp} health OK  — ping #{pings}", flush=True)
    else:
        print(f"   ⚠️ {stamp} health KO ({err}) — ping #{pings}", flush=True)